# ⚖️ Full Pipeline — KAGGLE (2×T4, LLM fp16 không quant)

Sinh `results.json` 2000 câu. I/O dùng `/kaggle/working`. LLM **Qwen2.5-7B fp16 trải đều 2×T4** (không 4-bit) → không OOM, chất lượng đầy đủ.

**Trước khi chạy (Settings bên phải):**
- `Accelerator` → **GPU T4 x2** (bắt buộc — fp16 7B ~15GB cần 2 GPU; 1 GPU sẽ tràn).
- `Internet` → **On** (tải HF model/dataset).
- Upload `corpus_articles.jsonl` + `stage1_questions.json` (+ `corpus_emb.npy`/`retrieved.json` nếu có): **Add Input → Upload** HOẶC kéo vào `/kaggle/working`.
- Chạy xong: `results.json` + `submission.zip` ở `/kaggle/working`, tải từ tab **Output**. **Save Version** để giữ giữa các phiên (resume).

## 1. Cài thư viện

In [ ]:
!pip install -q "sentence-transformers>=3.0" "transformers>=4.44" accelerate rank_bm25

## 2. Kiểm tra GPU

In [ ]:
!nvidia-smi -L

## 3. Tìm file đầu vào (/kaggle/input hoặc /kaggle/working)

In [ ]:
import os, glob, json
WORK = "/kaggle/working"

def find_file(name):
    cands = [os.path.join(WORK, name), name] + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for p in cands:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Không thấy {name}. Upload qua 'Add Input' hoặc đặt vào /kaggle/working.")

CORPUS_PATH = find_file("corpus_articles.jsonl")
QS_PATH = find_file("stage1_questions.json")
questions = json.load(open(QS_PATH, encoding="utf-8"))
print("corpus:", CORPUS_PATH)
print("questions:", QS_PATH, "→", len(questions), "câu")

## 4. Phase A — Retrieval (fp16 reranker, CAND=20, batch-embed). Lưu vào /kaggle/working, bỏ qua nếu đã có.

In [ ]:
import re, numpy as np, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi

EMBED_ID, RERANK_ID = "AITeamVN/Vietnamese_Embedding", "AITeamVN/Vietnamese_Reranker"
MAX_SEQ, RERANK_MAX, CAND, TOP_K, MIN_SCORE, MARGIN, RR_BATCH = 1024, 256, 20, 8, 0.0, 4.0, 32
DEV = "cuda" if torch.cuda.is_available() else "cpu"
RET_PATH = f"{WORK}/retrieved.json"
EMB_PATH = f"{WORK}/corpus_emb.npy"
def tok(t): return re.findall(r"\w+", (t or "").lower(), flags=re.UNICODE)

if os.path.exists(RET_PATH):
    cache = {int(k): v for k, v in json.load(open(RET_PATH, encoding="utf-8")).items()}
    print(f"Đã có retrieved.json ({len(cache)} câu) → BỎ QUA Phase A.")
else:
    corpus = [json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8")
              if l.strip() and json.loads(l).get("doc_number")]
    docs_text = [f"{r['title']}\n{r['text']}" for r in corpus]
    print("corpus:", len(corpus), "| device:", DEV)

    emb = SentenceTransformer(EMBED_ID, device=DEV); emb.max_seq_length = MAX_SEQ
    # corpus_emb có sẵn (upload/Save Version trước) → khỏi embed lại
    emb_in = EMB_PATH if os.path.exists(EMB_PATH) else (glob.glob("/kaggle/input/**/corpus_emb.npy", recursive=True) or [None])[0]
    if emb_in and os.path.exists(emb_in):
        corpus_emb = np.load(emb_in).astype("float32")
        assert len(corpus_emb) == len(corpus), "corpus_emb.npy không khớp corpus"
        print("Dùng corpus_emb.npy có sẵn → BỎ QUA embed corpus")
    else:
        corpus_emb = emb.encode(docs_text, batch_size=128, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True).astype("float32")
        np.save(EMB_PATH, corpus_emb); print("Đã lưu", EMB_PATH)
    q_emb = emb.encode([q["question"] for q in questions], batch_size=128, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype("float32")
    bm25 = BM25Okapi([tok(t) for t in docs_text])
    rr_tok = AutoTokenizer.from_pretrained(RERANK_ID)
    rr = AutoModelForSequenceClassification.from_pretrained(RERANK_ID).to(DEV).eval()
    if DEV == "cuda": rr = rr.half()

    @torch.no_grad()
    def rerank(query, idxs):
        pairs = [[query, docs_text[i]] for i in idxs]; out = []
        for j in range(0, len(pairs), RR_BATCH):
            enc = rr_tok(pairs[j:j+RR_BATCH], padding=True, truncation=True,
                         max_length=RERANK_MAX, return_tensors="pt").to(DEV)
            out.extend(rr(**enc).logits.view(-1).float().cpu().tolist())
        return out

    cache = {}
    for n, q in enumerate(questions):
        dense = np.argsort(corpus_emb @ q_emb[n])[::-1][:CAND]
        bm = bm25.get_scores(tok(q["question"])); sp = [i for i in np.argsort(bm)[::-1][:CAND] if bm[i] > 0]
        rrf = {}
        for rank, i in enumerate(dense): rrf[int(i)] = rrf.get(int(i), 0) + 1/(60+rank+1)
        for rank, i in enumerate(sp):    rrf[int(i)] = rrf.get(int(i), 0) + 1/(60+rank+1)
        fused = sorted(rrf, key=rrf.get, reverse=True)[:CAND]
        scored = sorted(zip(fused, rerank(q["question"], fused)), key=lambda x: x[1], reverse=True)
        seen, dd = set(), []
        for i, s in scored:
            k = (corpus[i]["doc_number"], corpus[i]["article"])
            if k not in seen: seen.add(k); dd.append((i, s))
        sel = []
        if dd:
            sel = [dd[0]]; top = dd[0][1]
            for i, s in dd[1:TOP_K]:
                if s < MIN_SCORE or s < top - MARGIN: break
                sel.append((i, s))
        cache[int(q["id"])] = [{k: corpus[i][k] for k in ("doc_number","clean_name","article","title","text")} for i, _ in sel]
        if (n+1) % 200 == 0: print(f"retrieved {n+1}/{len(questions)}")

    json.dump({str(k): v for k, v in cache.items()}, open(RET_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    print("Đã lưu", RET_PATH, "→ giải phóng VRAM Phase A")
    del emb, rr, corpus_emb, q_emb; import gc; gc.collect(); torch.cuda.empty_cache()

## 5. Phase B1a — Nạp LLM fp16 trải 2×T4 (CHẠY 1 LẦN; đừng chạy lại để khỏi nạp lại model)

In [ ]:
import gc, os, torch
for _v in ["emb", "rr", "rr_tok", "corpus_emb", "q_emb", "bm25"]:
    globals().pop(_v, None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoTokenizer, AutoModelForCausalLM
LLM_ID = "Qwen/Qwen2.5-7B-Instruct"   # hợp lệ <14B, Apache. (đổi 3B nếu muốn nhẹ hơn)
ltok = AutoTokenizer.from_pretrained(LLM_ID)
ltok.padding_side = "left"                                   # BẮT BUỘC cho batched generate
if ltok.pad_token is None: ltok.pad_token = ltok.eos_token
# fp16 KHÔNG quant — device_map='auto' tự trải ~15GB lên 2×T4 (no OOM, đủ chất lượng).
# Cần 2 GPU (Accelerator = GPU T4 x2). Nếu chỉ 1 GPU → tràn: đổi model 3B hoặc dùng 4-bit.
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16,
                                           device_map="auto", low_cpu_mem_usage=True)
llm.eval()
print(f"LLM fp16 loaded: {LLM_ID} | {torch.cuda.device_count()} GPU")
print("device_map:", getattr(llm, "hf_device_map", "single"))

## 5b. Phase B1b — Helpers + TEST 5 câu (chạy lại thoải mái, KHÔNG nạp lại model)

In [ ]:
import re, json, glob, os, gc, torch
WORK = globals().get("WORK", "/kaggle/working")
if "cache" not in globals():     # B1b tự nạp nếu chưa chạy Phase A trong session
    _rj = next((p for p in [f"{WORK}/retrieved.json"]
                + sorted(glob.glob("/kaggle/input/**/retrieved.json", recursive=True))
                if os.path.exists(p)), None)
    assert _rj, "Chưa có retrieved.json — chạy Phase A (cell 4) trước."
    cache = {int(k): v for k, v in json.load(open(_rj, encoding="utf-8")).items()}
    print("loaded cache:", len(cache), "from", _rj)
if "questions" not in globals():
    _qj = next((p for p in [f"{WORK}/stage1_questions.json"]
                + sorted(glob.glob("/kaggle/input/**/stage1_questions.json", recursive=True))
                if os.path.exists(p)), None)
    assert _qj, "Chưa có stage1_questions.json"
    questions = json.load(open(_qj, encoding="utf-8"))

def build_prompt(query, ctx):
    blocks = [f"[{i}] {d.get('clean_name','')} ({d.get('doc_number','')}) — {d.get('article','')}\n{d.get('text','')}"
              for i, d in enumerate(ctx, 1)]
    context_str = "\n\n".join(blocks) if blocks else "(Không có căn cứ.)"
    must = ", ".join(sorted({d.get("article", "") for d in ctx if d.get("article")}))
    system = ("Bạn là trợ lý pháp lý AI cho doanh nghiệp SME Việt Nam. Trả lời DỰA HOÀN TOÀN trên CƠ SỞ PHÁP LÝ; "
              "tuyệt đối không bịa, không suy diễn ngoài căn cứ.\nTốt = chính xác (số liệu/điều kiện/thời hạn lấy từ "
              "điều luật), đầy đủ, thực tiễn, rõ ràng dễ hiểu.\nBẮT BUỘC nêu rõ số Điều + tên văn bản cho mỗi căn cứ. "
              "Nếu không đủ căn cứ, nói rõ + khuyến nghị tham vấn luật sư.")
    user = f"CƠ SỞ PHÁP LÝ:\n{context_str}\n\nCÂU HỎI: {query}\n\nTrích dẫn rõ các điều: {must}.\nCâu trả lời:"
    return system, user

def _texts(items):
    out = []
    for q in items:
        s, u = build_prompt(q["question"], cache.get(int(q["id"]), []))
        out.append(ltok.apply_chat_template([{"role": "system", "content": s}, {"role": "user", "content": u}],
                                             add_generation_prompt=True, tokenize=False))
    return out

@torch.no_grad()
def _gen(texts, max_new=400):
    inputs = ltok(texts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=ltok.eos_token_id)
    res = [ltok.decode(o, skip_special_tokens=True).strip() for o in out[:, inputs.input_ids.shape[1]:]]
    del inputs, out
    return res

def gen_batch(items, max_new=400):
    """Sinh cả lô; nếu OOM → lùi về từng câu (không bao giờ chết cả lô)."""
    texts = _texts(items)
    try:
        return _gen(texts, max_new)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        res = []
        for t in texts:
            try: res.append(_gen([t], max_new)[0])
            except Exception: res.append(None)
            torch.cuda.empty_cache()
        return res

def build_fields(ctx):
    docs, arts = [], []
    for d in ctx:
        c, nm, a = d.get("doc_number", ""), d.get("clean_name", ""), d.get("article", "")
        if not c or not a: continue
        if f"{c}|{nm}" not in docs: docs.append(f"{c}|{nm}")
        if f"{c}|{nm}|{a}" not in arts: arts.append(f"{c}|{nm}|{a}")
    return docs, arts

def ensure_cit(ans, ctx):
    present = set(re.findall(r"Điều\s+(\d+)", ans)); miss = {}
    for d in ctx:
        m = re.match(r"Điều\s+(\d+)", d.get("article", ""))
        if m and m.group(1) not in present: miss.setdefault(d.get("clean_name", ""), []).append(d["article"])
    if not miss: return ans
    return ans.rstrip() + "\n\nCăn cứ pháp lý áp dụng: " + "; ".join(f"{', '.join(a)} ({nm})" for nm, a in miss.items()) + "."

# TEST 5 câu
for q, ans in zip(questions[:5], gen_batch(questions[:5])):
    ctx = cache.get(int(q["id"]), []); _, ra = build_fields(ctx)
    ans = ensure_cit(ans, ctx) if ctx else "Chưa tìm thấy căn cứ."
    print("=" * 90); print("Q%s: %s" % (q["id"], q["question"])); print("Điều:", ra); print(ans)
print("\n>>> Ổn thì chạy Phase B2.")

## 6. Phase B2 — FULL 2000 câu (BATCHED, checkpoint mỗi lô, resume). fp16 2×T4 ~2-4h.

In [ ]:
import json, os, torch
RESULTS = f"{WORK}/results.json"
BATCH = 8                                  # fp16 2×T4: 8 an toàn (gen_batch tự lùi từng câu nếu OOM). Dư thì thử 12.

results = []
if os.path.exists(RESULTS):
    try: results = json.load(open(RESULTS, encoding="utf-8"))
    except Exception: results = []
done = {r["id"] for r in results}
todo = [q for q in questions if int(q["id"]) not in done]
print(f"resume: {len(done)} xong | còn {len(todo)}")

for i in range(0, len(todo), BATCH):
    batch = todo[i:i+BATCH]
    answers = gen_batch(batch)             # tự lùi về từng câu nếu OOM → trả None cho câu lỗi
    for q, ans in zip(batch, answers):
        qid = int(q["id"]); ctx = cache.get(qid, []); rd, ra = build_fields(ctx)
        if not ctx:
            ans = "Chưa tìm thấy căn cứ pháp lý phù hợp. Khuyến nghị tham vấn luật sư."
        elif ans is None:
            ans = "Căn cứ pháp lý liên quan: " + "; ".join(ra)
        else:
            ans = ensure_cit(ans, ctx)
        results.append({"id": qid, "question": q["question"], "answer": ans,
                        "relevant_docs": rd, "relevant_articles": ra})
    json.dump(sorted(results, key=lambda r: r["id"]), open(RESULTS, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    torch.cuda.empty_cache()
    print(f"  {len(results)}/{len(questions)}")

results.sort(key=lambda r: r["id"])
json.dump(results, open(RESULTS, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("DONE:", len(results))

## 7. Đóng gói `submission.zip` (tải từ tab Output)

In [ ]:
import zipfile, json, os
RESULTS = f"{WORK}/results.json"
assert len(json.load(open(RESULTS))) == len(questions), "Thiếu câu!"
ZIP = f"{WORK}/submission.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RESULTS, arcname="results.json")     # flat: results.json ở gốc
print("✓", ZIP, "| tải từ tab Output (Data) bên phải.")

## Ghi chú Kaggle
- **fp16 2×T4 (không quant)**: model trải đều 2 GPU → không OOM, chất lượng đầy đủ. `gen_batch` tự lùi về từng câu nếu 1 lô bị spike → không bao giờ chết giữa chừng.
- **Session dài** (~12h, 30h GPU/tuần) → 2000 câu (~2-4h) chạy trọn 1 phiên.
- **Save Version** để giữ `retrieved.json` / `corpus_emb.npy` / `results.json` giữa các phiên (resume).
- Ngắt giữa chừng → mở lại, đảm bảo `/kaggle/working/results.json` còn → chạy lại Phase B2 sẽ resume.
- **Tại sao không vLLM:** image Kaggle hiện tại lệch CUDA-build (vllm cu13 vs torch cu12) → `libcudart.so.13` not found. transformers fp16 chắc ăn, khỏi vá.
- Cấu hình retrieval khớp local (top8/score≥0/margin4). Cần thì sửa `MIN_SCORE/MARGIN` ở Phase A.